# UnifyWeaver 家谱教程

本交互式笔记本演示了如何使用 UnifyWeaver 将 Prolog 谓词编译为 Bash 脚本。

## 前提条件

- 已安装 SWI-Prolog
- 可用 UnifyWeaver 库
- 已安装 Prolog Jupyter 内核 (`pip install prolog-jupyter-kernel`)

## 学习目标

完成本笔记本后，你将能够：
1. 定义 Prolog 事实与规则
2. 使用 UnifyWeaver 将谓词编译为 Bash
3. 测试生成的 Bash 脚本
4. 理解传递闭包的编译原理

## 步骤 1：初始化 UnifyWeaver 环境

首先，我们需要加载 UnifyWeaver 模块。我们将使用 education 目录下的 `init.pl` 文件。

In [ ]:
% Load the initialization file
['../init'].

## 步骤 2：定义家庭关系

让我们根据圣经家谱定义一些父母与子女的关系。

In [ ]:
% Define parent facts
:- dynamic parent/2.

parent(abraham, isaac).
parent(abraham, ishmael).
parent(isaac, jacob).
parent(isaac, esau).
parent(jacob, reuben).
parent(jacob, simeon).
parent(jacob, levi).
parent(jacob, judah).

## 步骤 3：测试父母关系查询

在编译之前，让我们用几个 Prolog 查询来验证数据是否正确。

In [ ]:
% Query: Who are Abraham's children?
parent(abraham, Child).

In [ ]:
% Query: Who are Jacob's children?
parent(jacob, Child).

## 步骤 4：定义祖先关系

现在我们来定义传递闭包 — `ancestor` 关系。

In [ ]:
% Define ancestor as transitive closure of parent
:- dynamic ancestor/2.

% Base case: parent is an ancestor
ancestor(X, Y) :- parent(X, Y).

% Recursive case: if X is parent of Y and Y is ancestor of Z, then X is ancestor of Z
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

## 步骤 5：测试祖先关系查询

验证我们的祖先谓词是否正常工作。

In [ ]:
% Query: Is Abraham an ancestor of Jacob?
( ancestor(abraham, jacob) ->
    writeln('Yes: Abraham is an ancestor of Jacob')
;
    writeln('No: Abraham is not an ancestor of Jacob')
).

In [ ]:
% Query: Who are all of Abraham's descendants?
ancestor(abraham, Descendant).

## 步骤 6：将 Parent 编译为 Bash

现在进入精彩部分 — 将我们的 `parent/2` 事实编译为一个 Bash 脚本！

In [ ]:
% Load the stream compiler
\+ \+ (
    use_module(unifyweaver(core/stream_compiler)),

    % Compile parent facts to bash
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    writeln('Generated Bash code for parent/2:'),
    writeln(_BashCode)
).

## 步骤 7：保存 Parent 脚本

将生成的 Bash 代码保存到文件中。

In [ ]:
% Save to file
\+ \+ (
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    setup_call_cleanup(
        open('../output/parent.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/parent.sh')
).

## 步骤 8：将 Ancestor 编译为 Bash

现在编译使用递归的 `ancestor/2` 谓词。

In [ ]:
% Load the recursive compiler
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),

    % Compile ancestor to bash
    compile_recursive(ancestor/2, [], _BashCode),
    writeln('Generated Bash code for ancestor/2:'),
    writeln(_BashCode)
).

## 步骤 9：保存 Ancestor 脚本

将祖先脚本保存到文件中。

In [ ]:
% Save to file
\+ \+ (
    compile_recursive(ancestor/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/ancestor.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/ancestor.sh')
).

## 步骤 10：测试生成的脚本

现在测试我们生成的 Bash 脚本！我们将使用 `%%bash` 魔法命令来执行 bash 命令。

In [ ]:
%%bash
# Source the parent script
source ../output/parent.sh

# Test: Who are Abraham's children?
echo "Abraham's children:"
parent abraham

In [ ]:
%%bash
# Source both scripts
source ../output/parent.sh
source ../output/ancestor.sh

# Test: Who are Abraham's descendants?
echo "Abraham's descendants:"
ancestor abraham

In [ ]:
%%bash
# Source both scripts
source ../output/parent.sh
source ../output/ancestor.sh

# Test: Is Abraham an ancestor of Judah?
if ancestor abraham judah >/dev/null 2>&1; then
    echo "✓ Yes, Abraham is an ancestor of Judah"
else
    echo "✗ No"
fi

## 步骤 11：理解编译策略

让我们分析 UnifyWeaver 的内部处理机制：

1. **Parent 编译**：使用 `stream_compiler` 创建简单的流式函数，输出所有父母-子女对

2. **Ancestor 编译**：检测传递闭包模式并应用 BFS（广度优先搜索）优化，以高效计算所有可达祖先

让我们检查具体的编译策略：

In [ ]:
% Check if ancestor is classified as recursive
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),
    recursive_compiler:classify_predicate(ancestor/2, _Classification),
    format('Ancestor classification: ~w~n', [_Classification])
).

## 总结

在本笔记本中，你学习了：

✅ 如何定义 Prolog 事实与规则

✅ 如何对事实使用 UnifyWeaver 的 `stream_compiler`

✅ 如何对递归谓词使用 UnifyWeaver 的 `recursive_compiler`

✅ 如何测试生成的 Bash 脚本

✅ UnifyWeaver 会自动检测传递闭包并应用 BFS 优化

## 后续步骤

尝试完成以下练习：

1. 向家谱中添加更多家庭成员
2. 定义 `grandparent/2` 谓词并进行编译
3. 创建 `sibling/2` 谓词（拥有共同父母的两人）
4. 查看生成的 Bash 代码以深入理解 BFS 算法

继续前往 **笔记本 2：递归模式比较**，学习高级递归模式！